# RL-ROI-Net Autonomous Kaggle Training (BIG plan, v3)

Self-driving **1-seed-gate then full 3-seed** temporal-region training on the
full FF++ pool (~7,964 train / 1,220 val / 1,216 test sequences), **60 epochs**
per seed, with **early stopping** (patience 12) so each seed stops once val AUC
stops improving (~15-20 epochs), and a **cross-set evaluation** after seed 0.

## What it does automatically
1. Reads `rlroinet_kaggle.zip` from the mounted Kaggle dataset `/kaggle/input`
2. Extracts code + weights + FF++ manifest data into `/kaggle/working` (local, no download)
3. Installs dependencies (torch is already preinstalled on Kaggle)
4. Patches the runtime code to the current versions (trainer, temporal_region, cross-set eval)
5. Pulls prior checkpoints/state from a private Kaggle dataset (resume across the 12h session cap)
6. Detects GPU: **T4 => fp16** (native tensor cores), Ampere+ => bf16
7. Runs **seed 0** alone on GPU 0; every metrics table now has a **PASS/FAIL**
   column against the paper acceptance bar + an OVERALL acceptance row
8. **Gate:** seed-0 max val video AUC >= 0.80 => next step; else FAILED + stop
9. **Cross-set eval:** FF++ per-method breakdown + Celeb-DF (if present) -> saved
   to `outputs/seed0/eval_cross_set.json`, printed before escalating
10. **Seeds 1+2 run in PARALLEL**, one per GPU (both T4s used, not just GPU 0)
11. Full run => `run_summary.json` (mean +/- std) + human-readable `RESULTS.md`

## Required before running
- **Accelerator: GPU T4 x2** and **Internet ON** (right sidebar -> Settings)
- The private Kaggle dataset `rlroinet-kaggle` with `rlroinet_kaggle.zip` inside
  (add it via the **Add Input** button or it is already linked to this notebook)
- Optional overrides: `RLROINET_SEEDS` (default `0 1 2`), `RLROINET_EPOCHS` (60),
  `RLROINET_GATE_AUC` (0.80), `RLROINET_EARLY_STOP_PATIENCE` (12),
  `RLROINET_SNAPSHOT_MINUTES` (30), `RLROINET_BUDGET_MINUTES` (540 = 9h hard quota),
  `RLROINET_CROSS_QUICK` (100 = Celeb-DF videos/class sampled for the cross-set check)


In [ ]:
# Cell 1 - Environment (run once per session)
import os, sys
from pathlib import Path

ZIP_NAME = os.environ.get("RLROINET_ZIP_NAME", "rlroinet_kaggle.zip")
DATASET_NAME = os.environ.get("RLROINET_DATASET", "rlroinet-kaggle")
WORK = Path(os.environ.get("RLROINET_WORK", "/kaggle/working"))

print("kaggle user:", os.environ.get("KAGGLE_USERNAME", "") or "(not set)")
print("torch will not be reinstalled (preinstalled on Kaggle)")

In [ ]:
# Cell 2 - Unpack the package from /kaggle/input (run once per session)
import shutil, os, zipfile
from pathlib import Path

WORK = Path(os.environ.get("RLROINET_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

INPUT = Path(os.environ.get("RLROINET_INPUT", "/kaggle/input"))
print("contents of", INPUT, ":")
if not INPUT.exists():
    print("  <does not exist>")
for p in sorted(INPUT.rglob("*"))[:50]:
    depth = len(p.parts) - len(INPUT.parts)
    print("  " * depth + p.name + ("/" if p.is_dir() else ""))

# Case 1: select the configured package only; never extract an arbitrary zip.
zip_candidates = [INPUT / DATASET_NAME / ZIP_NAME, INPUT / ZIP_NAME]
zip_candidates.extend(p for p in INPUT.rglob(ZIP_NAME) if p not in zip_candidates)
zip_path = next((p for p in zip_candidates if p.is_file()), None)
if zip_path is not None:
    print("found configured package zip:", zip_path)
    if not (WORK / "rlroinet" / "__init__.py").exists():
        print("extracting...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(WORK)
        print("extracted entries:", len(zipfile.ZipFile(zip_path).namelist()))
    else:
        print("already extracted; skipping")
# Case 2: the tree was uploaded already extracted -> copy it from the configured dataset.
elif (WORK / "rlroinet" / "__init__.py").exists():
    print("rlroinet already in WORK; skipping")
else:
    init_files = list((INPUT / DATASET_NAME).rglob("rlroinet/__init__.py")) \
        if (INPUT / DATASET_NAME).exists() else []
    if init_files:
        src_root = init_files[0].parent.parent
        print("found extracted tree at:", src_root)
        print("copying source tree and data into WORK...")
        for d in ("rlroinet", "outputs"):
            src = src_root / d
            if src.is_dir():
                shutil.copytree(src, WORK / d, dirs_exist_ok=True)
        for d in src_root.iterdir():
            if d.is_dir() and d.name in ("data",):
                shutil.copytree(d, WORK / d.name, dirs_exist_ok=True)
        for f in ("requirements.txt", "pyproject.toml"):
            src = src_root / f
            if src.is_file():
                shutil.copy2(src, WORK / f)
        print("copied tree entries:", len(list(WORK.rglob("*"))))
    else:
        raise SystemExit(
            f"no {ZIP_NAME!r} and no extracted rlroinet tree found under "
            f"{INPUT / DATASET_NAME}. Attach the {DATASET_NAME} dataset via Add Input."
        )

import rlroinet
print("rlroinet OK from", rlroinet.__file__)

In [ ]:
# Cell 3 - Repair + verify deps (idempotent; fixes a broken numpy if present)
import importlib, subprocess, sys

def _have(module_name):
    try:
        importlib.import_module(module_name)
        return True
    except Exception:
        return False

def _pip(args, label):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                        "--no-cache-dir", "--no-warn-conflicts", *args])
    if r.returncode != 0:
        raise SystemExit(f"{label} install failed (rc={r.returncode})")

# numpy/scipy/sklearn/cv2/matplotlib are ALL preinstalled and consistent on a
# FRESH Kaggle session. Only if one is broken do we uninstall the damaged stack
# and reinstall everything TOGETHER (no version pins) so the resolver picks a
# compatible matrix. The pip warnings about tpot/numba/umap etc. are for
# preinstalled packages we do not use -- safe to ignore.
STACK = ("numpy", "scipy", "sklearn", "cv2", "matplotlib")
MODS = ("numpy", "scipy", "sklearn", "cv2", "matplotlib")
need = [m for m in MODS if not _have(m)]
if need:
    print("broken/missing deps:", need)
    print("uninstalling the damaged stack, then reinstalling consistently...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                    "numpy", "scipy", "scikit-learn", "matplotlib",
                    "opencv-python", "opencv-python-headless"],
                   capture_output=True)
    _pip(["numpy", "scipy", "scikit-learn", "opencv-python-headless",
          "matplotlib"], "deps")
    still = [m for m in MODS if not _have(m)]
    if still:
        raise SystemExit(
            "Environment still broken after repair. Restart this session "
            "(top-right stop, then start again) to get a clean Kaggle image, "
            "then re-run cells 1-3."
        )
else:
    print("all deps healthy (numpy, scipy, sklearn, cv2, matplotlib); nothing to install")

# final verification
import cv2, sklearn, numpy
print("cv2", cv2.__version__, "| sklearn", sklearn.__version__, "| numpy", numpy.__version__)

In [ ]:
# Cell 4 - Write the autonomous driver to disk (base64: no escaping issues)
import base64
_B64 = "..."
from pathlib import Path
Path("/kaggle/working/kaggle_driver.py").write_bytes(base64.b64decode(_B64))
print("driver written:", len(base64.b64decode(_B64)), "chars")

In [ ]:
# Cell 4b - Patch temporal_region.py to the fixed version if stale (idempotent)
import base64
_B64 = "IiIiSm9pbnQgdGVtcG9yYWwgdmlkZW8gY2xhc3NpZmljYXRpb24gYW5kIHBlci1mcmFtZSBST0kgbG9jYWxpemF0aW9uLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCgpmcm9tIC5jb25maWcgaW1wb3J0IENvbmZpZwpmcm9tIC5yZWdpb25fYWdlbnQgaW1wb3J0IFJlZ2lvbkFnZW50CmZyb20gLnRlbXBvcmFsX3F1YWxpdHkgaW1wb3J0IFRlbXBvcmFsUXVhbGl0eUhlYWQKZnJvbSAudmVyZGljdCBpbXBvcnQgVmVyZGljdE1vZGVsCgoKZGVmIHF1YWxpdHlfZmVhdHVyZXMoZmFjZXM6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgIiIiUmV0dXJuIGx1bWluYW5jZSwgY29udHJhc3QsIGFuZCB2ZXJ0aWNhbC1kZXRhaWwgcXVhbGl0eSBwZXIgc2VxdWVuY2UgZnJhbWUuIiIiCiAgICBsdW1hID0gZmFjZXMubWVhbihkaW09MikKICAgIHNoYXJwbmVzcyA9IChsdW1hWzosIDosIDE6LCA6XSAtIGx1bWFbOiwgOiwgOi0xLCA6XSkuYWJzKCkubWVhbihkaW09KDIsIDMpKQogICAgcmV0dXJuIHRvcmNoLnN0YWNrKChsdW1hLm1lYW4oZGltPSgyLCAzKSksIGx1bWEuc3RkKGRpbT0oMiwgMyksIHVuYmlhc2VkPUZhbHNlKSwgc2hhcnBuZXNzKSwgZGltPS0xKQoKCmNsYXNzIFRlbXBvcmFsUmVnaW9uQWdlbnQobm4uTW9kdWxlKToKICAgICIiIlRyYWluIHRlbXBvcmFsIGRlY2lzaW9ucyB3aGlsZSBwcmVzZXJ2aW5nIG1hc2sgYW5kIGZvdXItUk9JIGV2aWRlbmNlLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjZmc6IENvbmZpZywgdmVyZGljdDogVmVyZGljdE1vZGVsKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNmZyA9IGNmZwogICAgICAgIHNlbGYuZnJhbWUgPSBSZWdpb25BZ2VudChjZmcsIHZlcmRpY3QsIHdpdGhfY2xhc3NpZmllcj1UcnVlKQogICAgICAgIHNlbGYudGVtcG9yYWwgPSBUZW1wb3JhbFF1YWxpdHlIZWFkKAogICAgICAgICAgICB2ZXJkaWN0LmZlYXR1cmVfY2hhbm5lbHMsIGNmZy5tb2RlbC50ZW1wb3JhbF9oaWRkZW4sIGNmZy5tb2RlbC50ZW1wb3JhbF9xdWFsaXR5X2hpZGRlbiwKICAgICAgICApCiAgICAgICAgc2VsZi5mcmFtZS5jbGFzc2lmaWVyX3RyYWluZWQgPSBUcnVlCgogICAgZGVmIGZvcndhcmQoc2VsZiwgZmFjZXM6IHRvcmNoLlRlbnNvcikgLT4gZGljdDoKICAgICAgICBiYXRjaCwgc3RlcHMgPSBmYWNlcy5zaGFwZVs6Ml0KICAgICAgICBmcmFtZV9vdXQgPSBzZWxmLmZyYW1lKGZhY2VzLmZsYXR0ZW4oMCwgMSkpCiAgICAgICAgcmV0dXJuIHNlbGYuX2NvbXBvc2UoZnJhbWVfb3V0LCBxdWFsaXR5X2ZlYXR1cmVzKGZhY2VzKSwgYmF0Y2gsIHN0ZXBzKQoKICAgIGRlZiBmb3J3YXJkX2Zyb21fZmVhdHVyZXMoc2VsZiwgZmFjZXM6IHRvcmNoLlRlbnNvciwgZmVhdDogdG9yY2guVGVuc29yLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICB2ZXJkaWN0X3Byb2JzOiB0b3JjaC5UZW5zb3IpIC0+IGRpY3Q6CiAgICAgICAgIiIiQ29tcG9zZSB0aGUgdGVtcG9yYWwgaGVhZCBmcm9tIGNhY2hlZCBiYWNrYm9uZSBmZWF0dXJlcy4KCiAgICAgICAgYGBmZWF0YGAgaXMgYGAoQipzdGVwcywgQywgaCwgdylgYCBhbmQgYGB2ZXJkaWN0X3Byb2JzYGAgaXMgYGAoQipzdGVwcywpYGAKICAgICAgICBhcyBwcm9kdWNlZCBieSA6ZnVuYzpgcmxyb2luZXQuZmVhdHVyZV9jYWNoZS5idWlsZF9mZWF0dXJlX2NhY2hlYCwgYWxpZ25lZAogICAgICAgIHRvIHRoZSBmbGF0dGVuZWQgc2VxdWVuY2UgZnJhbWVzLiBGYWNlcyBhcmUgc3RpbGwgbmVlZGVkIGZvciB0aGUKICAgICAgICBsaWdodHdlaWdodCBxdWFsaXR5IGZlYXR1cmVzIGJ1dCB0aGUgZnJvemVuIGJhY2tib25lIGlzIG5ldmVyIHJ1bi4KICAgICAgICAiIiIKICAgICAgICBiYXRjaCwgc3RlcHMgPSBmYWNlcy5zaGFwZVs6Ml0KICAgICAgICBxdWFsaXR5ID0gcXVhbGl0eV9mZWF0dXJlcyhmYWNlcykucmVzaGFwZSgtMSwgMykKICAgICAgICByZXR1cm4gc2VsZi5mb3J3YXJkX2Zyb21fY2FjaGVkKGZlYXQsIHZlcmRpY3RfcHJvYnMsIHF1YWxpdHksIGJhdGNoLCBzdGVwcykKCiAgICBkZWYgZm9yd2FyZF9mcm9tX2NhY2hlZChzZWxmLCBmZWF0OiB0b3JjaC5UZW5zb3IsIHZlcmRpY3RfcHJvYnM6IHRvcmNoLlRlbnNvciwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHF1YWxpdHk6IHRvcmNoLlRlbnNvciwgYmF0Y2g6IGludCwgc3RlcHM6IGludCkgLT4gZGljdDoKICAgICAgICAiIiJDb21wb3NlIHRoZSB0ZW1wb3JhbCBoZWFkIGZyb20gZnVsbHktY2FjaGVkIGlucHV0cyAobm8gZmFjZXMgbmVlZGVkKS4KCiAgICAgICAgYGBxdWFsaXR5YGAgaXMgYGAoQipzdGVwcywgMylgYCBpbWFnZS1xdWFsaXR5IGZlYXR1cmVzIHByZWNvbXB1dGVkIGJ5IHRoZQogICAgICAgIGNhY2hlOyB0aGlzIHBhdGggbmV2ZXIgdG91Y2hlcyBmYWNlIHRlbnNvcnMsIHNvIHRoZSB0cmFpbmluZyBsb29wIGlzIHB1cmUKICAgICAgICB0ZW5zb3Igc2h1ZmZsaW5nIGludG8gdGhlIHRyYWluYWJsZSBoZWFkcy4KICAgICAgICAiIiIKICAgICAgICBmcmFtZV9vdXQgPSBzZWxmLmZyYW1lLmZvcndhcmRfZnJvbV9mZWF0dXJlcyhmZWF0LCB2ZXJkaWN0X3Byb2JzKQogICAgICAgIHJldHVybiBzZWxmLl9jb21wb3NlKGZyYW1lX291dCwgcXVhbGl0eSwgYmF0Y2gsIHN0ZXBzKQoKICAgIGRlZiBfY29tcG9zZShzZWxmLCBmcmFtZV9vdXQ6IGRpY3QsIHF1YWxpdHk6IHRvcmNoLlRlbnNvciwgYmF0Y2g6IGludCwgc3RlcHM6IGludCkgLT4gZGljdDoKICAgICAgICAiIiJBc3NlbWJsZSB0aGUgdGVtcG9yYWwgaGVhZCBvdXRwdXQgZnJvbSBhIGZyYW1lIGFnZW50J3MgZm9yd2FyZCBwYXNzLgoKICAgICAgICBgYGZyYW1lX291dGBgIGlzIHRoZSBkaWN0IGZyb20gOm1ldGg6YFJlZ2lvbkFnZW50LmZvcndhcmRgIChtYXNrLCByZWdpb24sCiAgICAgICAgZmVhdHVyZXMsIGNsc19sb2dpdHMpIG92ZXIgZmxhdHRlbmVkIGBgKEIqc3RlcHMsIC4uLilgYCBmYWNlcywgYW5kCiAgICAgICAgYGBxdWFsaXR5YGAgaXMgYGAoQipzdGVwcywgMylgYCBwZXItZnJhbWUgcXVhbGl0eSBmZWF0dXJlcy4gU2hhcmVkIGJ5CiAgICAgICAgdGhlIGNhY2hlZCBhbmQgdW5jYWNoZWQgZm9yd2FyZCBwYXRocyBzbyBib3RoIHByb2R1Y2UgaWRlbnRpY2FsIG91dHB1dHMuCiAgICAgICAgIiIiCiAgICAgICAgZmVhdHVyZXMgPSBzZWxmLmZyYW1lLmdhcChmcmFtZV9vdXRbImZlYXR1cmVzIl0pLmZsYXR0ZW4oMSkucmVzaGFwZShiYXRjaCwgc3RlcHMsIC0xKQogICAgICAgIGZyYW1lX2xvZ2l0cyA9IGZyYW1lX291dFsiY2xzX2xvZ2l0cyJdLnJlc2hhcGUoYmF0Y2gsIHN0ZXBzKQogICAgICAgIHRlbXBvcmFsID0gc2VsZi50ZW1wb3JhbChmZWF0dXJlcywgcXVhbGl0eS5yZXNoYXBlKGJhdGNoLCBzdGVwcywgLTEpLCBmcmFtZV9sb2dpdHMpCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInZpZGVvX2xvZ2l0cyI6IHRlbXBvcmFsWyJsb2dpdHMiXSwKICAgICAgICAgICAgImF0dGVudGlvbiI6IHRlbXBvcmFsWyJhdHRlbnRpb24iXSwKICAgICAgICAgICAgImZyYW1lX2xvZ2l0cyI6IGZyYW1lX2xvZ2l0cywKICAgICAgICAgICAgIm1hc2siOiBmcmFtZV9vdXRbIm1hc2siXS5yZXNoYXBlKGJhdGNoLCBzdGVwcywgKmZyYW1lX291dFsibWFzayJdLnNoYXBlWzE6XSksCiAgICAgICAgICAgICJyZWdpb24iOiBmcmFtZV9vdXRbInJlZ2lvbiJdLnJlc2hhcGUoYmF0Y2gsIHN0ZXBzLCAqZnJhbWVfb3V0WyJyZWdpb24iXS5zaGFwZVsxOl0pLAogICAgICAgIH0KCiAgICBkZWYgdHJhaW4oc2VsZiwgbW9kZTogYm9vbCA9IFRydWUpOgogICAgICAgIHN1cGVyKCkudHJhaW4obW9kZSkKICAgICAgICBzZWxmLmZyYW1lLnRyYWluKG1vZGUpCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgdHJhaW5hYmxlX3BhcmFtZXRlcnMoc2VsZik6CiAgICAgICAgcmV0dXJuIFsqc2VsZi5mcmFtZS50cmFpbmFibGVfcGFyYW1ldGVycygpLCAqc2VsZi50ZW1wb3JhbC5wYXJhbWV0ZXJzKCldCg=="
from pathlib import Path
from importlib import import_module
_p = Path("/kaggle/working/rlroinet/temporal_region.py")
if "def _compose" in _p.read_text():
    print("temporal_region.py already fixed; skipping")
else:
    _p.write_bytes(base64.b64decode(_B64))
    print("patched temporal_region.py with the fixed _compose version")
import importlib as _il
import rlroinet.temporal_region as _tr
_il.reload(_tr)
print("_compose present:", hasattr(_tr.TemporalRegionAgent, "_compose"))

In [ ]:
# Cell 4c - Patch trainer + cross-set eval to the current versions (idempotent)
import base64
from pathlib import Path
_ROOT = Path("/kaggle/working/rlroinet")
_TRAIN = "..."
_CROSS = "IiIiUG9zdC10cmFpbmluZyBjcm9zcy1zZXQgZXZhbHVhdGlvbiBmb3IgYSB0ZW1wb3JhbC1yZWdpb24gdjIgY2hlY2twb2ludC4KClJ1bnMgYXV0b21hdGljYWxseSBhZnRlciBzZWVkIDAgcGFzc2VzIHRoZSBoZWFsdGggZ2F0ZSAoYmVmb3JlIHNlZWRzIDErMiwgc28gYQpicm9rZW4gdHJhbnNmZXIgaXMgY2F1Z2h0IGJlZm9yZSBtb3JlIEdQVSBob3VycyBhcmUgc3BlbnQpLiBUd28gY2hlY2tzOgoKICAxLiAqKlVuc2Vlbi1tZXRob2QgRkYrKyBicmVha2Rvd24qKiAtLSB0aGUgaGVsZC1vdXQgRkYrKyB0ZXN0IHZpZGVvcyBncm91cGVkIGJ5CiAgICAgbWFuaXB1bGF0aW9uIG1ldGhvZCAoRmFjZVN3YXAgLyBEZWVwZmFrZXMgLyBGYWNlMkZhY2UgLyBOZXVyYWxUZXh0dXJlcykuCiAgICAgVGhpcyBpcyB0aGUgc3Ryb25nZXN0IHNpZ25hbCBvZiB3aGV0aGVyIHRoZSBmcm96ZW4gYGBob25pMDVgYCBiYWNrYm9uZQogICAgICh0cmFpbmVkIG9uIENlbGViLURGIHYyKSBnZW5lcmFsaXplcyBhY3Jvc3MgbWFuaXB1bGF0aW9uIG1ldGhvZHMuCiAgMi4gKipVbnNlZW4tZGF0YXNldCBDZWxlYi1ERioqIC0tIHJhdyBDZWxlYi1ERiB2aWRlb3Mgc2NvcmVkIHZlcmRpY3Qtb25seSwKICAgICB3aGVuIGEgYGBkYXRhL2NlbGViZGZgYCB0cmVlIGlzIHByZXNlbnQuCgpSZXBvcnQ6IGBgPHNlZWRfZGlyPi9ldmFsX2Nyb3NzX3NldC5qc29uYGAgKGFsc28gcHJpbnRlZCB0byBzdGRvdXQpLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGYxX3Njb3JlLCByb2NfYXVjX3Njb3JlCgpmcm9tIC5jb25maWcgaW1wb3J0IENvbmZpZywgZGVmYXVsdF9jb25maWcKZnJvbSAuZGF0YSBpbXBvcnQgUkVHSU9OX05BTUVTLCBsb2FkX2luZGV4LCByb2lfbWFzawpmcm9tIC5kYXRhLnRlbXBvcmFsX2RhdGEgaW1wb3J0IFNlcXVlbmNlTWFza0RhdGFzZXQKZnJvbSAuZXZhbHVhdGUgaW1wb3J0IGNhbGlicmF0aW9uX2N1cnZlLCBlZXIKZnJvbSAucHJlY2lzaW9uIGltcG9ydCBhdXRvY2FzdCwgZW5hYmxlX3RmMzIKZnJvbSAucHJlZGljdCBpbXBvcnQgYWdncmVnYXRlX3ZpZGVvX2NvbmZpZGVuY2UKZnJvbSAudGVtcG9yYWxfcmVnaW9uIGltcG9ydCBUZW1wb3JhbFJlZ2lvbkFnZW50CmZyb20gLnRyYWluX3RlbXBvcmFsX3JlZ2lvbiBpbXBvcnQgX2lvdSwgX21ha2VfbG9hZGVyLCBfbWV0cmljcwpmcm9tIC52ZXJkaWN0IGltcG9ydCBsb2FkX3ZlcmRpY3QKCmxvZyA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJybHJvaW5ldC5ldmFsX2Nyb3NzX3NldCIpCgpfQ0hFQ0tQT0lOVF9GT1JNQVQgPSAidGVtcG9yYWwtcmVnaW9uLXYyIgpfVkVSRElDVCA9ICJob25pMDUiCgoKZGVmIF9sb2FkX2FnZW50KGNoZWNrcG9pbnQ6IFBhdGgsIGRldmljZTogc3RyKSAtPiB0dXBsZToKICAgIGNrcHQgPSB0b3JjaC5sb2FkKGNoZWNrcG9pbnQsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PVRydWUpCiAgICBpZiBub3QgaXNpbnN0YW5jZShja3B0LCBkaWN0KSBvciBja3B0LmdldCgiY2hlY2twb2ludF9mb3JtYXQiKSAhPSBfQ0hFQ0tQT0lOVF9GT1JNQVQ6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmImV4cGVjdGVkIGEge19DSEVDS1BPSU5UX0ZPUk1BVH0gY2hlY2twb2ludCwgZ290IHtjaGVja3BvaW50fSIpCiAgICBjZmcgPSBDb25maWcuZnJvbV9kaWN0KGNrcHRbImNmZyJdKSBpZiBpc2luc3RhbmNlKGNrcHQuZ2V0KCJjZmciKSwgZGljdCkgZWxzZSBkZWZhdWx0X2NvbmZpZygpCiAgICBjZmcudHJhaW4uZGV2aWNlID0gZGV2aWNlCiAgICB2ZXJkaWN0ID0gbG9hZF92ZXJkaWN0KF9WRVJESUNULCBkZXZpY2U9ZGV2aWNlKQogICAgYWdlbnQgPSBUZW1wb3JhbFJlZ2lvbkFnZW50KGNmZywgdmVyZGljdCkudG8oZGV2aWNlKQogICAgYWdlbnQuZnJhbWUucmVnaW9uLmxvYWRfc3RhdGVfZGljdChja3B0WyJyZWdpb25fc3RhdGUiXSkKICAgIGFnZW50LmZyYW1lLmNsYXNzaWZpZXIubG9hZF9zdGF0ZV9kaWN0KGNrcHRbImNsYXNzaWZpZXJfc3RhdGUiXSkKICAgIGFnZW50LnRlbXBvcmFsLmxvYWRfc3RhdGVfZGljdChja3B0WyJ0ZW1wb3JhbF9zdGF0ZSJdKQogICAgcmV0dXJuIGFnZW50LCBjZmcsIGNrcHQKCgpAdG9yY2guaW5mZXJlbmNlX21vZGUoKQpkZWYgX2NvbGxlY3QoYWdlbnQsIGxvYWRlciwgY2ZnLCB0ZW1wZXJhdHVyZTogZmxvYXQgPSAxLjApOgogICAgIiIiT25lIHBhc3Mgb3ZlciBhIFNlcXVlbmNlTWFza0RhdGFzZXQ6IG1ldHJpY3MgKyBwZXItdmlkZW8gbGFiZWxzL3Njb3Jlcy4iIiIKICAgIGFnZW50LmV2YWwoKQogICAgdmlkZW9zLCBsYWJlbHMsIHNjb3JlcywgaW91cywgaGl0cywgZmFrZV9mcmFtZXMgPSBbXSwgW10sIFtdLCBbXSwgMCwgMAogICAgcm9pX3ZhbHVlcyA9IHt9CiAgICBmb3IgYmF0Y2ggaW4gbG9hZGVyOgogICAgICAgIGZhY2VzID0gYmF0Y2hbImZhY2VzIl0udG8oY2ZnLnRyYWluLmRldmljZSkKICAgICAgICB3aXRoIGF1dG9jYXN0KGNmZyk6CiAgICAgICAgICAgIG91dCA9IGFnZW50KGZhY2VzKQogICAgICAgIHNjb3Jlcy5leHRlbmQodG9yY2guc2lnbW9pZChvdXRbInZpZGVvX2xvZ2l0cyJdLnNxdWVlemUoLTEpLmZsb2F0KCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAvIHRlbXBlcmF0dXJlKS5jcHUoKS50b2xpc3QoKSkKICAgICAgICBsYWJlbHMuZXh0ZW5kKGJhdGNoWyJsYWJlbHMiXS50byh0b3JjaC5pbnQ2NCkudG9saXN0KCkpCiAgICAgICAgdmlkZW9zLmV4dGVuZChiYXRjaFsidmlkZW9zIl0pCiAgICAgICAgaWYgZmxvYXQoYmF0Y2hbImxhYmVscyJdLnN1bSgpKSA8IDE6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbWFza3MgPSBiYXRjaFsibWFza3MiXQogICAgICAgIHByZWRpY3RlZCA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwuaW50ZXJwb2xhdGUoCiAgICAgICAgICAgIG91dFsibWFzayJdLmZsYXR0ZW4oMCwgMSkuZmxvYXQoKSwgc2l6ZT1tYXNrcy5zaGFwZVstMjpdLAogICAgICAgICAgICBtb2RlPSJiaWxpbmVhciIsIGFsaWduX2Nvcm5lcnM9RmFsc2UpLnNxdWVlemUoMSkuY3B1KCkubnVtcHkoKQogICAgICAgIGZvciBzYW1wbGVfaW5kZXgsIGxhYmVsIGluIGVudW1lcmF0ZShiYXRjaFsibGFiZWxzIl0udG9saXN0KCkpOgogICAgICAgICAgICBpZiBpbnQobGFiZWwpICE9IDE6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBsYXlvdXRzID0gYmF0Y2hbImxheW91dHMiXVtzYW1wbGVfaW5kZXhdCiAgICAgICAgICAgIGZvciBzdGVwLCBsYXlvdXQgaW4gZW51bWVyYXRlKGxheW91dHMpOgogICAgICAgICAgICAgICAgb2Zmc2V0ID0gc2FtcGxlX2luZGV4ICogbGVuKGxheW91dHMpICsgc3RlcAogICAgICAgICAgICAgICAgcHJlZCwgdGFyZ2V0ID0gcHJlZGljdGVkW29mZnNldF0sIG1hc2tzW3NhbXBsZV9pbmRleCwgc3RlcF0ubnVtcHkoKQogICAgICAgICAgICAgICAgaW91cy5hcHBlbmQoX2lvdShwcmVkLCB0YXJnZXQpKQogICAgICAgICAgICAgICAgaGl0cyArPSBpbnQobnAubG9naWNhbF9hbmQocHJlZCA+IDAuNSwgdGFyZ2V0ID4gMC41KS5hbnkoKSkKICAgICAgICAgICAgICAgIGZha2VfZnJhbWVzICs9IDEKICAgICAgICAgICAgICAgIGZvciByZWdpb25faWQsIHJvaSBpbiByb2lfbWFzayhsYXlvdXQsIGNmZy5kYXRhLmZhY2Vfc2l6ZSkuaXRlbXMoKToKICAgICAgICAgICAgICAgICAgICByb2lfdmFsdWVzLnNldGRlZmF1bHQoUkVHSU9OX05BTUVTW3JlZ2lvbl9pZF0sIFtdKS5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgICAgIF9pb3UocHJlZCAqIG5wLmFzYXJyYXkocm9pKSwgdGFyZ2V0ICogbnAuYXNhcnJheShyb2kpKSkKICAgIG1ldHJpY3MgPSBfbWV0cmljcyhsYWJlbHMsIHNjb3JlcywgY2ZnKQogICAgbWV0cmljcy51cGRhdGUoewogICAgICAgICJyZWdpb25faW91IjogZmxvYXQobnAubWVhbihpb3VzKSkgaWYgaW91cyBlbHNlIGZsb2F0KCJuYW4iKSwKICAgICAgICAicmVnaW9uX2hpdCI6IGZsb2F0KGhpdHMgLyBtYXgoMSwgZmFrZV9mcmFtZXMpKSwKICAgICAgICAicGVyX3JvaV9pb3UiOiB7bmFtZTogZmxvYXQobnAubWVhbih2KSkgaWYgdiBlbHNlIGZsb2F0KCJuYW4iKQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgbmFtZSwgdiBpbiByb2lfdmFsdWVzLml0ZW1zKCl9LAogICAgICAgICJ0ZW1wZXJhdHVyZSI6IGZsb2F0KHRlbXBlcmF0dXJlKSwKICAgIH0pCiAgICByZXR1cm4gbWV0cmljcywgdmlkZW9zLCBucC5hc2FycmF5KGxhYmVscywgZHR5cGU9bnAuaW50NjQpLCBucC5hc2FycmF5KHNjb3JlcywgZHR5cGU9bnAuZmxvYXQzMikKCgpkZWYgX3Blcl9tZXRob2QodmlkZW9zLCBsYWJlbHMsIHNjb3JlcywgY2ZnKSAtPiBkaWN0OgogICAgIiIiR3JvdXAgdGhlIHNpbmdsZSBoZWxkLW91dCBGRisrIHRlc3Qgc2V0IGJ5IG1hbmlwdWxhdGlvbiBtZXRob2QuIiIiCiAgICByZWFsID0gW2kgZm9yIGksIF8gaW4gZW51bWVyYXRlKHZpZGVvcykgaWYgaW50KGxhYmVsc1tpXSkgPT0gMF0KICAgIGZha2VzID0gW2kgZm9yIGksIF8gaW4gZW51bWVyYXRlKHZpZGVvcykgaWYgaW50KGxhYmVsc1tpXSkgPT0gMV0KICAgIG1ldGhvZHMgPSBzb3J0ZWQoe3N0cih2aWRlb3NbaV0pLnNwbGl0KCIvIilbMF0gZm9yIGkgaW4gZmFrZXN9KQogICAgb3V0ID0ge30KICAgIGZvciBtIGluIG1ldGhvZHM6CiAgICAgICAgaWR4ID0gcmVhbCArIFtpIGZvciBpIGluIGZha2VzIGlmIHN0cih2aWRlb3NbaV0pLnNwbGl0KCIvIilbMF0gPT0gbV0KICAgICAgICBpZiBub3QgaWR4OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG1ldHJpY3MgPSBfbWV0cmljcyhsYWJlbHNbaWR4XSwgc2NvcmVzW2lkeF0sIGNmZykKICAgICAgICBtZXRyaWNzLnVwZGF0ZSh7Im1ldGhvZCI6IG0sICJuIjogbGVuKGlkeCksICJuX3JlYWwiOiBsZW4ocmVhbCksCiAgICAgICAgICAgICAgICAgICAgICAgICJuX2Zha2UiOiBsZW4oaWR4KSAtIGxlbihyZWFsKX0pCiAgICAgICAgb3V0W21dID0gbWV0cmljcwogICAgICAgIGxvZy5pbmZvKCJGRisrIG1ldGhvZCAlLTE2cyBBQ0M9JS40ZiBBVUM9JS40ZiBGMT0lLjRmIEVFUj0lLjRmIChuPSVkIGZha2U9JWQgcmVhbD0lZCkiLAogICAgICAgICAgICAgICAgIG0sIG1ldHJpY3NbImFjYyJdLCBtZXRyaWNzWyJhdWMiXSwgbWV0cmljc1siZjEiXSwgbWV0cmljc1siZWVyIl0sCiAgICAgICAgICAgICAgICAgbWV0cmljc1sibiJdLCBtZXRyaWNzWyJuX2Zha2UiXSwgbWV0cmljc1sibl9yZWFsIl0pCiAgICByZXR1cm4gb3V0CgoKQHRvcmNoLmluZmVyZW5jZV9tb2RlKCkKZGVmIF9jZWxlYmRmKGFnZW50LCBjZmcsIGNlbGViZGZfZGlyLCBkZXZpY2UsIHF1aWNrOiBpbnQgPSAwLCB0ZW1wZXJhdHVyZTogZmxvYXQgPSAxLjApIC0+IGRpY3Q6CiAgICAiIiJDcm9zcy1kYXRhc2V0IHZlcmRpY3Qtb25seSBjaGVjayBvbiByYXcgQ2VsZWItREYgdmlkZW9zIChubyBHVCBtYXNrcykuIiIiCiAgICBmcm9tIC5nZW5lcmFsaXplIGltcG9ydCBfbGlzdF92aWRlb3MsIF9zYW1wbGVfZnJhbWVzCiAgICB2aWRlb3MgPSBfbGlzdF92aWRlb3MoUGF0aChjZWxlYmRmX2RpcikpCiAgICBpZiBxdWljayA+IDA6CiAgICAgICAgcmVhbCA9IFt2IGZvciB2IGluIHZpZGVvcyBpZiB2WyJsYWJlbCJdID09IDBdCiAgICAgICAgZmFrZSA9IFt2IGZvciB2IGluIHZpZGVvcyBpZiB2WyJsYWJlbCJdID09IDFdCiAgICAgICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApCiAgICAgICAgcm5nLnNodWZmbGUocmVhbCkKICAgICAgICBybmcuc2h1ZmZsZShmYWtlKQogICAgICAgIHZpZGVvcyA9IHJlYWxbOnF1aWNrXSArIGZha2VbOnF1aWNrXQogICAgbGFiZWxzLCBzY29yZXMsIG1pc3NpbmcgPSBbXSwgW10sIDAKICAgIGFnZW50LmV2YWwoKQogICAgZm9yIGl0ZW0gaW4gdmlkZW9zOgogICAgICAgIGZyYW1lcyA9IF9zYW1wbGVfZnJhbWVzKGl0ZW1bInZpZGVvIl0sIGNmZykKICAgICAgICBpZiBmcmFtZXMgaXMgTm9uZToKICAgICAgICAgICAgbWlzc2luZyArPSAxCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgd2l0aCBhdXRvY2FzdChjZmcpOgogICAgICAgICAgICBvdXQgPSBhZ2VudChmcmFtZXMudW5zcXVlZXplKDApLnRvKGRldmljZSkpCiAgICAgICAgcHJvYiA9IHRvcmNoLnNpZ21vaWQob3V0WyJ2aWRlb19sb2dpdHMiXS5zcXVlZXplKC0xKS5mbG9hdCgpIC8gdGVtcGVyYXR1cmUpCiAgICAgICAgZnJhbWVfc2NvcmVzID0gcHJvYi5yZXNoYXBlKC0xKS5jcHUoKS5udW1weSgpCiAgICAgICAgc2NvcmVzLmFwcGVuZChmbG9hdChhZ2dyZWdhdGVfdmlkZW9fY29uZmlkZW5jZShmcmFtZV9zY29yZXMsIGNmZykpKQogICAgICAgIGxhYmVscy5hcHBlbmQoaXRlbVsibGFiZWwiXSkKICAgIGxhYmVsc19hID0gbnAuYXNhcnJheShsYWJlbHMpCiAgICBzY29yZXNfYSA9IG5wLmFzYXJyYXkoc2NvcmVzKQogICAgcHJlZHMgPSAoc2NvcmVzX2EgPj0gY2ZnLmV2YWwudGhyZXNob2xkKS5hc3R5cGUoaW50KQogICAgaGFzX2JvdGggPSBsZW4obnAudW5pcXVlKGxhYmVsc19hKSkgPiAxCiAgICBtZXRyaWNzID0gewogICAgICAgICJkYXRhc2V0IjogImNlbGViZGZfcmF3IiwKICAgICAgICAibl92aWRlb3MiOiBpbnQobGVuKGxhYmVsc19hKSksCiAgICAgICAgIm5fcmVhbCI6IGludCgobGFiZWxzX2EgPT0gMCkuc3VtKCkpLAogICAgICAgICJuX2Zha2UiOiBpbnQoKGxhYmVsc19hID09IDEpLnN1bSgpKSwKICAgICAgICAic2tpcHBlZCI6IG1pc3NpbmcsCiAgICAgICAgImFjYyI6IGZsb2F0KChwcmVkcyA9PSBsYWJlbHNfYSkubWVhbigpKSBpZiBsZW4obGFiZWxzX2EpIGVsc2UgZmxvYXQoIm5hbiIpLAogICAgICAgICJhdWMiOiBmbG9hdChyb2NfYXVjX3Njb3JlKGxhYmVsc19hLCBzY29yZXNfYSkpIGlmIGhhc19ib3RoIGVsc2UgZmxvYXQoIm5hbiIpLAogICAgICAgICJmMSI6IGZsb2F0KGYxX3Njb3JlKGxhYmVsc19hLCBwcmVkcywgemVyb19kaXZpc2lvbj0wKSkgaWYgaGFzX2JvdGggZWxzZSBmbG9hdCgibmFuIiksCiAgICAgICAgImVlciI6IGVlcihsYWJlbHNfYSwgc2NvcmVzX2EpIGlmIGxlbihsYWJlbHNfYSkgZWxzZSBmbG9hdCgibmFuIiksCiAgICAgICAgImVjZSI6IChjYWxpYnJhdGlvbl9jdXJ2ZShsYWJlbHNfYSwgc2NvcmVzX2EsIGNmZy5ldmFsLmNhbGlicmF0aW9uX2JpbnMpCiAgICAgICAgICAgICAgICBpZiBsZW4obGFiZWxzX2EpIGVsc2UgZmxvYXQoIm5hbiIpKSwKICAgIH0KICAgIGxvZy5pbmZvKCJDZWxlYi1ERiBjcm9zcy1kYXRhc2V0OiBBQ0M9JS40ZiBBVUM9JS40ZiBGMT0lLjRmIEVFUj0lLjRmIChuPSVkIHJlYWw9JWQgZmFrZT0lZCkiLAogICAgICAgICAgICAgbWV0cmljc1siYWNjIl0sIG1ldHJpY3NbImF1YyJdLCBtZXRyaWNzWyJmMSJdLCBtZXRyaWNzWyJlZXIiXSwKICAgICAgICAgICAgIG1ldHJpY3NbIm5fdmlkZW9zIl0sIG1ldHJpY3NbIm5fcmVhbCJdLCBtZXRyaWNzWyJuX2Zha2UiXSkKICAgIHJldHVybiBtZXRyaWNzCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IkNyb3NzLXNldCBldmFsdWF0aW9uIGZvciB0ZW1wb3JhbC1yZWdpb24gdjIiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNoZWNrcG9pbnQiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJwYXRoIHRvIHNlZWQgYmVzdC5wdCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZmZwcC1kaXIiLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY2VsZWJkZi1kaXIiLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcXVpY2siLCB0eXBlPWludCwgZGVmYXVsdD0wLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImNhcCByYXcgQ2VsZWItREYgdmlkZW9zIHBlciBjbGFzcyAoMCA9IHVubGltaXRlZCkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWFtcC1kdHlwZSIsIGRlZmF1bHQ9ImJmMTYiLCBjaG9pY2VzPVsiZnAxNiIsICJiZjE2Il0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgZGVmYXVsdD0iY3VkYSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0ib3V0cHV0cy9ldmFsX2Nyb3NzX3NldC5qc29uIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS13b3JrZXJzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NCkKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKICAgIGxvZ2dpbmcuYmFzaWNDb25maWcobGV2ZWw9bG9nZ2luZy5JTkZPLCBmb3JtYXQ9IiUobGV2ZWxuYW1lKXMgJShtZXNzYWdlKXMiKQogICAgaWYgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgb3Igbm90IHN0cihhcmdzLmRldmljZSkuc3RhcnRzd2l0aCgiY3VkYSIpOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoImNyb3NzLXNldCBldmFsdWF0aW9uIHJlcXVpcmVzIENVREEiKQogICAgaWYgYXJncy5hbXBfZHR5cGUgPT0gImJmMTYiIGFuZCBub3QgdG9yY2guY3VkYS5pc19iZjE2X3N1cHBvcnRlZCgpOgogICAgICAgIGxvZy53YXJuaW5nKCJiZjE2IG5vdCBuYXRpdmVseSBzdXBwb3J0ZWQgb24gdGhpcyBHUFU7IGZhbGxpbmcgYmFjayB0byBmcDE2IikKICAgICAgICBhcmdzLmFtcF9kdHlwZSA9ICJmcDE2IgogICAgZW5hYmxlX3RmMzIoVHJ1ZSkKCiAgICBhZ2VudCwgY2ZnLCBja3B0ID0gX2xvYWRfYWdlbnQoUGF0aChhcmdzLmNoZWNrcG9pbnQpLCBhcmdzLmRldmljZSkKICAgIGNmZy50cmFpbi5hbXBfZHR5cGUgPSBhcmdzLmFtcF9kdHlwZQogICAgY2ZnLnRyYWluLmRldmljZSA9IGFyZ3MuZGV2aWNlCiAgICBpZiBhcmdzLmZmcHBfZGlyOgogICAgICAgIGNmZy5kYXRhLmZmcHBfZGlyID0gYXJncy5mZnBwX2RpcgogICAgY2ZnLnJlc29sdmVfcGF0aHMoKQoKICAgIGxvZy5pbmZvKCItLS0gdW5zZWVuIG1ldGhvZHMgKEZGKysgaGVsZC1vdXQgdGVzdCwgcGVyLW1ldGhvZCkgLS0tIikKICAgIHRlc3RfaW5kZXggPSBsb2FkX2luZGV4KGNmZywgInRlc3QiKQogICAgdGVzdF9kcyA9IFNlcXVlbmNlTWFza0RhdGFzZXQodGVzdF9pbmRleCwgY2ZnKQogICAgdGVzdF9sb2FkZXIgPSBfbWFrZV9sb2FkZXIodGVzdF9kcywgOCwgRmFsc2UsIGFyZ3Mud29ya2VycywgMikKICAgIG92ZXJhbGwsIHZpZGVvcywgbGFiZWxzLCBzY29yZXMgPSBfY29sbGVjdChhZ2VudCwgdGVzdF9sb2FkZXIsIGNmZykKICAgIHBlcl9tZXRob2QgPSBfcGVyX21ldGhvZCh2aWRlb3MsIGxhYmVscywgc2NvcmVzLCBjZmcpCiAgICBsb2cuaW5mbygiRkYrKyB1bnNlZW4tdmlkZW8gdGVzdDogQUNDPSUuNGYgQVVDPSUuNGYgRjE9JS40ZiByZWdpb25Jb1U9JS40ZiByZWdpb25IaXQ9JS40ZiAobj0lZCkiLAogICAgICAgICAgICAgb3ZlcmFsbFsiYWNjIl0sIG92ZXJhbGxbImF1YyJdLCBvdmVyYWxsWyJmMSJdLAogICAgICAgICAgICAgb3ZlcmFsbC5nZXQoInJlZ2lvbl9pb3UiLCBmbG9hdCgibmFuIikpLCBvdmVyYWxsLmdldCgicmVnaW9uX2hpdCIsIGZsb2F0KCJuYW4iKSksCiAgICAgICAgICAgICBvdmVyYWxsWyJuIl0pCgogICAgbG9nLmluZm8oIi0tLSB1bnNlZW4gZGF0YXNldCAocmF3IENlbGViLURGLCB2ZXJkaWN0IG9ubHkpIC0tLSIpCiAgICBjZWxlYmRmX21ldHJpY3MsIGNlbGViZGZfc3RhdHVzID0gTm9uZSwgIm5vdCBydW46IG5vIC0tY2VsZWJkZi1kaXIiCiAgICBjZWxlYmRmX3Jvb3QgPSBQYXRoKGFyZ3MuY2VsZWJkZl9kaXIpIGlmIGFyZ3MuY2VsZWJkZl9kaXIgZWxzZSBOb25lCiAgICBpZiBjZWxlYmRmX3Jvb3QgaXMgTm9uZToKICAgICAgICBjZWxlYmRmX3Jvb3QgPSBQYXRoKGNmZy5kYXRhLmNlbGViZGZfZGlyKQogICAgaWYgY2VsZWJkZl9yb290IGFuZCBjZWxlYmRmX3Jvb3QuZXhpc3RzKCk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBjZWxlYmRmX21ldHJpY3MgPSBfY2VsZWJkZihhZ2VudCwgY2ZnLCBjZWxlYmRmX3Jvb3QsIGFyZ3MuZGV2aWNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBxdWljaz1hcmdzLnF1aWNrKQogICAgICAgICAgICBjZWxlYmRmX3N0YXR1cyA9ICJvayIKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgbG9nLmVycm9yKCJDZWxlYi1ERiBldmFsdWF0aW9uIGZhaWxlZDogJXMiLCBleGMpCiAgICAgICAgICAgIGNlbGViZGZfc3RhdHVzID0gZiJmYWlsZWQ6IHtleGN9IgogICAgZWxzZToKICAgICAgICBjZWxlYmRmX3N0YXR1cyA9IGYibm90IHJ1bjogbm8gQ2VsZWItREYgdHJlZSBhdCB7Y2VsZWJkZl9yb290fSIKCiAgICBwZXJfbWV0aG9kX2F1Y3MgPSBbbVsiYXVjIl0gZm9yIG0gaW4gcGVyX21ldGhvZC52YWx1ZXMoKQogICAgICAgICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UobS5nZXQoImF1YyIpLCAoaW50LCBmbG9hdCkpIGFuZCBtWyJhdWMiXSA9PSBtWyJhdWMiXV0KICAgIHN1bW1hcnkgPSB7CiAgICAgICAgIm92ZXJhbGxfdGVzdF9hdWMiOiBvdmVyYWxsLmdldCgiYXVjIiksCiAgICAgICAgIm92ZXJhbGxfdGVzdF9hY2MiOiBvdmVyYWxsLmdldCgiYWNjIiksCiAgICAgICAgIm92ZXJhbGxfdGVzdF9yZWdpb25faW91Ijogb3ZlcmFsbC5nZXQoInJlZ2lvbl9pb3UiKSwKICAgICAgICAicGVyX21ldGhvZF9tZWFuX2F1YyI6IHJvdW5kKGZsb2F0KG5wLm1lYW4ocGVyX21ldGhvZF9hdWNzKSksIDQpIGlmIHBlcl9tZXRob2RfYXVjcyBlbHNlIE5vbmUsCiAgICAgICAgInBlcl9tZXRob2RfbWluX2F1YyI6IHJvdW5kKGZsb2F0KG5wLm1pbihwZXJfbWV0aG9kX2F1Y3MpKSwgNCkgaWYgcGVyX21ldGhvZF9hdWNzIGVsc2UgTm9uZSwKICAgICAgICAicGVyX21ldGhvZF9uIjogbGVuKHBlcl9tZXRob2QpLAogICAgICAgICJjZWxlYmRmX2F1YyI6IGNlbGViZGZfbWV0cmljcy5nZXQoImF1YyIpIGlmIGNlbGViZGZfbWV0cmljcyBlbHNlIE5vbmUsCiAgICAgICAgImNlbGViZGZfYWNjIjogY2VsZWJkZl9tZXRyaWNzLmdldCgiYWNjIikgaWYgY2VsZWJkZl9tZXRyaWNzIGVsc2UgTm9uZSwKICAgICAgICAiY2VsZWJkZl9zdGF0dXMiOiBjZWxlYmRmX3N0YXR1cywKICAgIH0KICAgIHJlcG9ydCA9IHsKICAgICAgICAiY2hlY2twb2ludCI6IGFyZ3MuY2hlY2twb2ludCwKICAgICAgICAiZm9ybWF0IjogX0NIRUNLUE9JTlRfRk9STUFULAogICAgICAgICJ2ZXJkaWN0IjogX1ZFUkRJQ1QsCiAgICAgICAgImVwb2NoIjogY2twdC5nZXQoImVwb2NoIiksCiAgICAgICAgIm92ZXJhbGxfdGVzdCI6IG92ZXJhbGwsCiAgICAgICAgInBlcl9tZXRob2QiOiBwZXJfbWV0aG9kLAogICAgICAgICJjZWxlYmRmIjogY2VsZWJkZl9tZXRyaWNzLAogICAgICAgICJjZWxlYmRmX3N0YXR1cyI6IGNlbGViZGZfc3RhdHVzLAogICAgICAgICJzdW1tYXJ5Ijogc3VtbWFyeSwKICAgIH0KICAgIG91dCA9IFBhdGgoYXJncy5vdXQpCiAgICBvdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG91dC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LCBpbmRlbnQ9MiwgYWxsb3dfbmFuPVRydWUsIGRlZmF1bHQ9c3RyKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIGxvZy5pbmZvKCJjcm9zcy1zZXQgcmVwb3J0IHNhdmVkIHRvICVzIiwgb3V0KQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0yLCBhbGxvd19uYW49VHJ1ZSwgZGVmYXVsdD1zdHIpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"
_tp = _ROOT / "train_temporal_region.py"
_txt = _tp.read_text(encoding="utf-8") if _tp.exists() else ""
if "def _compose" in _txt and "PASS_BAR" in _txt and "early_stop_patience" in _txt:
    print("train_temporal_region.py already current; skipping")
else:
    _tp.write_bytes(base64.b64decode(_TRAIN))
    print("patched train_temporal_region.py (pass/fail + early stopping)")
_cp = _ROOT / "eval_cross_set.py"
_ctxt = _cp.read_text(encoding="utf-8") if _cp.exists() else ""
if "_CHECKPOINT_FORMAT" in _ctxt:
    print("eval_cross_set.py already current; skipping")
else:
    _cp.write_bytes(base64.b64decode(_CROSS))
    print("wrote eval_cross_set.py (cross-set evaluation)")


In [ ]:
# Cell 5 - RUN (autonomous driver; walk away after starting this cell)
import sys, os, importlib
sys.path.insert(0, "/kaggle/working")
import kaggle_driver
importlib.reload(kaggle_driver)
kaggle_driver.main()

## After the run
The driver ends with `DONE`. Download your results from `/kaggle/working/outputs`:
- `RESULTS.md` — human-readable, per-metric **PASS/FAIL** vs the acceptance bar + mean +/- std
- `run_summary.json` — machine-readable aggregate
- `seed0|1|2/test_metrics.json`, `decision_policy.json`, `run_config.json`, `metrics.json`
- `seed0/eval_cross_set.json` — FF++ per-method + Celeb-DF cross-set evaluation
Run the cell below to bundle them into one zip for download.

In [ ]:
# Cell 6 - Bundle results for download (run after the driver reports DONE)
import os, shutil, tempfile
from pathlib import Path

WORK = Path(os.environ.get("RLROINET_WORK", "/kaggle/working"))
source = WORK / "outputs"
out = WORK / "results_download.zip"
if out.exists():
    out.unlink()

# Explicit allowlist: no feature_cache, no arbitrary files, and no per-epoch
# checkpoint snapshots. Diagnostics are retained recursively for auditability.
root_files = {"RESULTS.md", "run_summary.json"}
seed_files = {"test_metrics.json", "decision_policy.json", "run_config.json",
              "metrics.json", "eval_cross_set.json"}
with tempfile.TemporaryDirectory(dir=WORK) as staging_name:
    staging = Path(staging_name) / "outputs"
    staging.mkdir()
    for name in root_files:
        src = source / name
        if src.is_file():
            shutil.copy2(src, staging / name)
    for src in sorted(source.glob("seed*")):
        if not src.is_dir():
            continue
        rel_seed = staging / src.name
        for path in src.rglob("*"):
            rel = path.relative_to(src)
            if path.is_file() and (
                rel.name in seed_files
                or (len(rel.parts) == 2 and rel.parts[0] == "checkpoints"
                    and rel.name in {"best.pt", "final.pt"})
                or "diagnostics" in rel.parts
            ):
                destination = rel_seed / rel
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, destination)
    diagnostics = source / "diagnostics"
    if diagnostics.is_dir():
        shutil.copytree(diagnostics, staging / "diagnostics", dirs_exist_ok=True)
    shutil.make_archive(str(out).with_suffix(""), "zip", staging)
print("results bundle:", out, f"({out.stat().st_size/1e6:.1f} MB) - download from the output panel")